In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from typing import Optional

device = "cuda"

In [15]:
from sklearn.model_selection import train_test_split
from datasets import load_dataset

nmt_original_valid_set, nmt_test_set = load_dataset(
    path="ageron/tatoeba_mt_train", name="eng-spa", split=["validation", "test"]
)
split = nmt_original_valid_set.train_test_split(train_size=0.9, seed=42)
nmt_train_set, nmt_valid_set = split["train"], split["test"]

In [16]:
nmt_train_set[0]

{'source_text': 'The two teams debated on the issue of nuclear power.',
 'target_text': 'Los dos equipos debatieron sobre el tema de la energía nuclear.',
 'source_lang': 'eng',
 'target_lang': 'spa'}

In [ ]:
import tokenizers
import tokenizers.pre_tokenizers
import tokenizers.trainers
from collections import namedtuple
from torch.utils.data import DataLoader

def train_eng_spa():
    for pair in nmt_train_set:
        yield pair["source_text"]
        yield pair["target_text"]

max_length = 256
vocab_size = 10000
nmt_tokenizer_model = tokenizers.models.BPE(unk_token="<unk>")
nmt_tokenizer = tokenizers.Tokenizer(nmt_tokenizer_model)
nmt_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
nmt_tokenizer.enable_truncation(max_length=max_length)
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
nmt_tokenizer_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=vocab_size, special_tokens=["<pad>", "<unk>", "<s>", "</s>"])
nmt_tokenizer.train_from_iterator(train_eng_spa(), nmt_tokenizer_trainer)

fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"]
class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device):
        return NmtPair(self.src_token_ids.to(device), self.src_mask.to(device),
                       self.tgt_token_ids.to(device), self.tgt_mask.to(device))

def nmt_collate_fn(batch):
    src_texts = [pair['source_text'] for pair in batch]
    tgt_texts = [f"<s> {pair['target_text']} </s>" for pair in batch]
    src_encodings = nmt_tokenizer.encode_batch(src_texts)
    tgt_encodings = nmt_tokenizer.encode_batch(tgt_texts)
    src_token_ids = torch.tensor([enc.ids for enc in src_encodings])
    tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings])
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings]) #padding mask
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings]) #padding mask
    inputs = NmtPair(src_token_ids, src_mask, tgt_token_ids[:,:-1], tgt_mask[:,:-1])
    labels = tgt_token_ids[:, 1:]
    return inputs, labels

batch_size = 32
nmt_train_loader = DataLoader(nmt_train_set, batch_size=batch_size, collate_fn=nmt_collate_fn,shuffle=True, pin_memory=True)
nmt_valid_loader = DataLoader(nmt_valid_set, batch_size=batch_size, collate_fn=nmt_collate_fn, pin_memory=True)
nmt_test_loader = DataLoader(nmt_test_set, batch_size=batch_size, collate_fn=nmt_collate_fn, pin_memory=True)

In [18]:
class PositionalEmbedding(nn.Module):
    def __init__(self, max_length, embed_dim, dropout=0.1):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(max_length, embed_dim) * 0.02)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, X):
        return self.dropout(X + self.pos_embed[:X.size(1)])

In [19]:
class MultiheadAttention(nn.Module):
    def __init__(self, embed_dim:int, num_heads:int, dropout:float = 0.1):
        super().__init__()
        self.h = num_heads
        self.d = embed_dim // num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
    
    def split_heads(self, X:Tensor):
        return X.view(X.size(0), X.size(1), self.h, self.d).transpose(1,2)
    
    def forward(self, query:Tensor, key:Tensor, value:Tensor, 
                attn_mask:Optional[Tensor]=None, 
                key_padding_mask:Optional[Tensor]=None) -> tuple[Tensor, Tensor]:
        q = self.split_heads(self.q_proj(query))  # (B, h, Lq, d)
        k = self.split_heads(self.k_proj(key))  # (B, h, Lk, d)
        v = self.split_heads(self.v_proj(value))  # (B, h, Lv, d) with Lv=Lk
        
        scores = q @ k.transpose(2, 3) / self.d**0.5  # (B, h, Lq, Lk)
        if attn_mask:
            scores = scores.masked_fill(attn_mask, -torch.inf)
        if key_padding_mask:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask, -torch.inf)
        weights = scores.softmax(dim=-1)  # (B, h, Lq, Lk)
        
        Z = self.dropout(weights) @ v  # (B, h, Lq, d)
        Z = Z.transpose(1, 2)  # (B, Lq, h, d)
        Z = Z.reshape(Z.size(0), Z.size(1), self.h * self.d)  # (B, Lq, h × d)
        
        return (self.out_proj(Z), weights)  # (B, Lq, h × d)

In [20]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model:int, nhead:int, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiheadAttention(d_model, nhead, dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, src:Tensor, src_mask:Optional[Tensor] = None, 
                src_key_padding_mask:Optional[Tensor]=None):
        attn, _ = self.self_attn(src, src, src, attn_mask=src_mask, key_padding_mask=src_key_padding_mask)
        Z = self.norm1(src + self.dropout(attn))
        ff = self.dropout(self.linear2(self.dropout(self.linear1(Z).relu())))
        return self.norm2(Z + ff)

In [21]:
class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model:int, nhead:int, dim_feedforward:int=2048, dropout:float=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.self_attn = MultiheadAttention(d_model, nhead, dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.norm1 = nn.LayerNorm(d_model)
        self.multihead_attn = MultiheadAttention(d_model, nhead, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.linear2 = nn.Linear(d_model, dim_feedforward)
        self.norm3 = nn.LayerNorm(d_model)
    
    def forward(self, tgt, memory:Tensor, tgt_mask:Optional[Tensor]=None, memory_mask:Optional[Tensor]=None,
                tgt_key_padding_mask:Optional[Tensor]=None, memory_key_padding_mask:Optional[Tensor]=None):
        attn1, _ = self.self_attn(tgt, tgt, tgt, attn_mask=tgt_mask, key_padding_mask=memory_key_padding_mask)
        Z = self.norm1(tgt + self.linear1(attn1))
        attn2, _ = self.multihead_attn(Z, memory, memory, attn_mask=tgt_mask, key_padding_mask=memory_key_padding_mask)
        Z = self.norm2(Z + self.dropout(attn2))
        ff = self.dropout(self.linear2(self.dropout(self.linear1(Z).relu())))
        return self.norm3(Z + ff)


In [29]:
class NmtTransformer(nn.Module):
    def __init__(self, vocab_size, max_length, embed_dim=512, pad_id=0, num_heads=8, num_layers=6, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.pos_embed = PositionalEmbedding(max_length, embed_dim, dropout)
        self.transformer = nn.Transformer(
            embed_dim, num_heads, num_encoder_layers=num_layers, num_decoder_layers=num_layers, batch_first=True)
        self.output = nn.Linear(embed_dim, vocab_size)
         
    def forward(self, pair):
        src_embeds = self.pos_embed(self.embed(pair.src_token_ids))
        tgt_embeds = self.pos_embed(self.embed(pair.tgt_token_ids))
        src_pad_mask = ~pair.src_mask.bool() #in pytorch, true is mask, so i flip with ~
        tgt_pad_mask = ~pair.tgt_mask.bool()
        size = [pair.tgt_token_ids.size(1)] * 2
        full_mask = torch.full(size, True, device=tgt_pad_mask.device)
        causal_mask = torch.triu(full_mask, diagonal=1)
        out_decoder = self.transformer(src_embeds, tgt_embeds, src_key_padding_mask=src_pad_mask,
                                       memory_key_padding_mask=src_pad_mask,
                                       tgt_mask=causal_mask, tgt_is_causal=True,
                                       tgt_key_padding_mask=tgt_pad_mask)
        return self.output(out_decoder).permute(0,2,1)

In [ ]:
model = NmtTransformer(vocab_size, max_length, embed_dim=128, pad_id=0, num_heads=4, num_layers=2, dropout=0.1).to(device)
optimizer = torch.optim.NAdam(model.parameters(), lr=0.002)
xentropy = nn.CrossEntropyLoss(ignore_index=0)

def train_epoch(model:nn.Module, loader:DataLoader, loss_fn:nn.Module, optimizer:torch.optim.Optimizer):
    model.train()
    total_loss = 0
    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        logits = model(inputs)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    return total_loss / len(loader)


def eval_epoch(model:nn.Module, loader:DataLoader, loss_fn:nn.Module):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            logits = model(inputs)
            loss = loss_fn(logits, labels)
            total_loss += loss.item()
    
    return total_loss / len(loader)


def translate(model, src_text, max_length=20, pad_id=0, eos_id=3):
    tgt_text = ""
    output_ids = []
    for index in range(max_length):
        batch, _ = nmt_collate_fn([{"source_text": src_text, "target_text": tgt_text}])
        with torch.no_grad():
            Y_logits = model(batch.to(device))
            Y_token_ids = Y_logits.argmax(dim=1)
            next_token_id = Y_token_ids[0, index].item()

        if next_token_id == eos_id:
            break
        output_ids.append(next_token_id)
        next_token = nmt_tokenizer.id_to_token(next_token_id)
        tgt_text += " " + next_token

    return nmt_tokenizer.decode(output_ids)

In [30]:
# training loop
n_epochs = 3
for epoch in range(n_epochs):
    train_loss = train_epoch(model, nmt_train_loader, xentropy, optimizer)
    valid_loss = eval_epoch(model, nmt_valid_loader, xentropy)
    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, valid_loss={valid_loss:.4f}")

current loss: 9.3601, avg loss: 0.001686512886940896
current loss: 8.5978, avg loss: 0.003235657666180585
current loss: 8.4017, avg loss: 0.0047494694134136576
current loss: 8.1175, avg loss: 0.0062120910163398265
current loss: 7.9794, avg loss: 0.007649829709852064
current loss: 7.6880, avg loss: 0.009035055830671981
current loss: 7.5805, avg loss: 0.010400913341625316
current loss: 7.5901, avg loss: 0.011768496917174744
current loss: 7.3632, avg loss: 0.01309520455094071
current loss: 7.1697, avg loss: 0.01438704963202949
current loss: 6.9264, avg loss: 0.015635051899128134
current loss: 6.6906, avg loss: 0.016840567030348218
current loss: 6.9245, avg loss: 0.01808823301985457
current loss: 6.5323, avg loss: 0.019265223580437738
current loss: 6.7525, avg loss: 0.020481885875667537
current loss: 6.3999, avg loss: 0.02163502615851325
current loss: 6.2295, avg loss: 0.02275746199461791
current loss: 6.4230, avg loss: 0.02391476734264477
current loss: 6.2542, avg loss: 0.0250416476447303

In [31]:
model.eval()
translate(model, "I like to play soccer with my friends at the beach")

'Me gusta jugar al fútbol con mis amigos en el playa .'

In [32]:
torch.save(model.state_dict(), "nmt_transformer.pth")
# model = NmtTransformer(vocab_size, max_length, embed_dim=128, pad_id=0, num_heads=4, num_layers=2).to(device)
# model.load_state_dict(torch.load("nmt_transformer.pth"))

In [33]:
import gc

del model, optimizer, xentropy
del nmt_train_loader, nmt_valid_loader, nmt_test_loader

gc.collect()

torch.cuda.empty_cache()